# Build a CONUS jobs file for `tools/run_batch.sh`

Generates the `drv basalt start_year tf experiment` lines for a **full national
run** of one experiment under one deployment scenario.

**Inputs**: Takes each point's feedstock source state and first application year 
from the published pickles associated with the main text (Beerling et al. 2025),
as well as `upstream/weathering_model/run_strc_6.mat`. The notebook collapses 
those points to the set of implied distinct column simulations.

Each line is one job, simulating one soil column. It's defined by:

| | field | comes from |
|---|---|---|
| climate + soil | `drv` | the point's driver cluster (1-based index) |
| rock | `basalt` | `<source state>_1` / `_2` — mineralogy only |
| timing | `start_year` | the point's first application year; drivers are calendar-sliced |

Note that the deployment window always runs to 2070, the last year in
the published record, so **`tf = 2071 - start_year`**.

**Requires** the same two inputs as `tools/compare_to_published.py`: the restored
`upstream/weathering_model/run_strc_6.mat` (437 clusters — see
[how-to-run.md § 2](../how-to-run.md)) and the USA_ERW deposit.

## 1. Configuration

`SCENARIO` refers to the 1 Gt or 2 Gt/yr case from the original paper (`S1_*`,
and `S2_*`) in the published pickles. 

In [1]:
from pathlib import Path
import sys, collections, pickle
import numpy as np
import pandas as pd

REPO      = Path.cwd().parent if Path.cwd().name == "tools" else Path.cwd()
TOOLS     = REPO / "tools"
RUN_STRC  = REPO / "upstream" / "weathering_model" / "run_strc_6.mat"
USA_ERW   = Path("~/Downloads/USA_ERW").expanduser()     # UPDATE to wherever it is on your machine
PICKLE    = USA_ERW / "data" / "compiled_run_CO2_a_b_v6_varz.pkl"
WEIGHTS   = REPO / "data" / "mineralogy_cluster_counts.csv"

SCENARIO   = 2       # 1 -> 1 Gt/yr,  2 -> 2 Gt/yr
EXPERIMENT = "E0"    # the only configuration run
LAST_YEAR  = 2070    # last year of the published record; sets tf = LAST_YEAR + 1 - start

for p in (RUN_STRC, PICKLE, WEIGHTS):
    assert p.exists(), f"missing: {p}"
print(f"repo      {REPO}\nscenario  S{SCENARIO} ({SCENARIO} Gt/yr)\nexperiment {EXPERIMENT}")

repo      /Users/tylerkukla/Documents/GitHub/beerling-2025-model-extension
scenario  S2 (2 Gt/yr)
experiment E0


## 2. Cluster lookup

`(crop, lind) -> drv`, imported from pickles. Note that `drv` index is 1-based.

In [2]:
sys.path.insert(0, str(TOOLS))
from compare_to_published import load_clusters, unwrap   
import scipy.io as sio                                   

key2drv = load_clusters(str(RUN_STRC))          # warns if not the 437-cluster file
mylist  = sio.loadmat(str(RUN_STRC), struct_as_record=False)["mylist"][0]
drv_crop = {i + 1: str(np.atleast_1d(unwrap(e).Crop).ravel()[0]) for i, e in enumerate(mylist)}

print(f"{len(mylist)} clusters, {len(key2drv)} (crop, lind) points mapped")

437 clusters, 5483 (crop, lind) points mapped


## 3. Collapse points to distinct simulations

Every deployed point is defined by `(drv, source state, start year)`. Points that
share these keys are run as a single simulation. 

Points with `bas = 'NaN'` are not treated under this scenario and are dropped.

In [3]:
with open(PICKLE, "rb") as fh:
    db = pickle.load(fh)

rows, members = [], {}
for v in db.values():
    src = str(np.atleast_1d(v["bas"]).ravel()[SCENARIO - 1])
    if src == "NaN":                      # not deployed in this scenario
        continue
    crop = str(np.atleast_1d(v["nam"]).ravel()[0])
    drv  = key2drv[(crop, int(v["lind"]))]
    yr   = int(np.atleast_1d(v["yy"]).ravel()[SCENARIO - 1])
    members.setdefault((drv, src, yr), []).append((v["nam_l"], float(v["ar"])))

for (drv, src, yr), pts in members.items():
    by_state = collections.Counter()
    for state, area in pts:
        by_state[state] += area
    rows.append(dict(
        drv=drv, source=src, start_year=yr, tf=LAST_YEAR + 1 - yr,
        basalt=src.replace(" ", "_"), crop=drv_crop[drv],
        n_points=len(pts), area_ha=sum(a for _, a in pts),
        recipients=tuple(sorted(by_state)), by_state=dict(by_state),
    ))

jobs = pd.DataFrame(rows).sort_values(["tf", "drv"], ascending=[False, True]).reset_index(drop=True)
print(f"{len(db):,} points -> {len(jobs)} distinct columns -> {2 * len(jobs)} runs (x2 mineralogies)")
jobs.drop(columns=["by_state"]).head()

5,483 points -> 338 distinct columns -> 676 runs (x2 mineralogies)


,drv,source,start_year,tf,basalt,crop,n_points,area_ha,recipients
0,23,Minnesota,2020,51,Minnesota,Temperate Maize,11,76278.378980,"(Wisconsin,)"
1,73,Wisconsin,2020,51,Wisconsin,Temperate Maize,6,60344.845010,"(Minnesota,)"
2,84,Michigan,2020,51,Michigan,Temperate Irrigated Maize,2,7260.888127,"(Wisconsin,)"
3,148,Michigan,2020,51,Michigan,Temperate Soy,5,34308.294860,"(Minnesota, Wisconsin)"
4,169,Minnesota,2021,50,Minnesota,Temperate Soy,2,8268.588977,"(Minnesota,)"


## 4. Sanity checks

Five checks before writing the job file:

1. **Scenario identity** — total points and treated area must match the table in
   [S 6.5](../how-to-run.md): 2,322 points / 24.5 M ha for 1 Gt, 3,904 / 49.2 M ha
   for 2 Gt.
2. **`tf` is in range** — the drivers only span 2020–2070.
3. **One deployment per cluster** — confirm each cluster has exactly one `(source, start_year)`. 
4. **Mineralogy weights exist** — every source state needs a `w` in
   `data/mineralogy_cluster_counts.csv` to mix `_1`/`_2` back together.
5. **No output-name collisions** — make sure we get only one `.mat` path per run.

In [4]:
EXPECTED = {1: (2322, 24.5e6), 2: (3904, 49.2e6)}      # how-to-run.md 6.5

n_pts, area = int(jobs.n_points.sum()), jobs.area_ha.sum()
exp_pts, exp_area = EXPECTED[SCENARIO]
print(f"1. points {n_pts:,} (expect {exp_pts:,})   area {area/1e6:.1f} M ha (expect {exp_area/1e6:.1f})")
assert n_pts == exp_pts, "point count does not match the published scenario -- wrong SCENARIO index?"
assert abs(area - exp_area) / exp_area < 0.01

assert jobs.tf.between(1, 51).all(), "tf outside the 2020-2070 driver window"
print(f"2. tf range {jobs.tf.min()}-{jobs.tf.max()} yr, {jobs.tf.sum():,} simulated years "
      f"per mineralogy ({2 * jobs.tf.sum():,} in total)")

dupes = jobs.drv.value_counts()
assert (dupes == 1).all(), f"clusters with >1 deployment: {dupes[dupes > 1].to_dict()}"
print(f"3. {len(jobs)} clusters, one deployment each")

w = pd.read_csv(WEIGHTS, comment="#")
missing = sorted(set(jobs.source) - set(w.state))
assert not missing, f"no mineralogy weight for: {missing}"
print(f"4. mineralogy weights present for all {jobs.source.nunique()} source states")

paths = [f"{EXPERIMENT}_drv{r.drv}_{r.basalt}_{m}_{r.tf}yr.mat"
         for r in jobs.itertuples() for m in (1, 2)]
assert len(set(paths)) == len(paths), "duplicate result filenames"
print(f"5. {len(paths)} distinct result filenames")

1. points 3,904 (expect 3,904)   area 49.2 M ha (expect 49.2)
2. tf range 3-51 yr, 8,773 simulated years per mineralogy (17,546 in total)
3. 338 clusters, one deployment each
4. mineralogy weights present for all 13 source states
5. 676 distinct result filenames


## 5. Pick which states to run

Two different state lists exist and they mean different things
([§ 6.3](../how-to-run.md)):

- **recipient state** — where the rock is spread. Sets climate and soil through
  the point's cluster. 
- **source state** — where the rock was quarried. Sets **mineralogy only**.

The filter below is on *recipient* state by default.

In [5]:
recip = collections.Counter()
recip_area = collections.Counter()
for r in jobs.itertuples():
    for s in r.recipients:
        recip[s] += 1
        recip_area[s] += r.area_ha

menu = (pd.DataFrame({"columns_touching_state": pd.Series(recip),
                      "area_ha": pd.Series(recip_area)})
          .sort_values("columns_touching_state", ascending=False))
menu.index.name = "recipient state"

span = (jobs.recipients.map(len) > 1).sum()
print(f"{len(menu)} recipient states; {span} of {len(jobs)} columns span more than one "
      f"(so the counts below sum to more than {len(jobs)})\n")
print(f"source states: {', '.join(sorted(jobs.source.unique()))}")
menu

20 recipient states; 224 of 338 columns span more than one (so the counts below sum to more than 338)

source states: California, Connecticut, Idaho, Massachusetts, Michigan, Minnesota, New Jersey, North Carolina, Oregon, Pennsylvania, Virginia, Washington, Wisconsin


,columns_touching_state,area_ha
recipient state,,
Illinois,94,1.785991e+07
Iowa,75,1.449055e+07
Indiana,70,1.259695e+07
Missouri,67,1.092617e+07
Kansas,57,7.114013e+06
Wisconsin,48,6.430016e+06
Nebraska,47,6.172191e+06
Michigan,45,6.187649e+06
Ohio,44,7.974734e+06


In [6]:
RECIPIENT_STATES = "all"     # "all", or e.g. ["Iowa", "Illinois"]
SOURCE_STATES    = "all"     # "all", or e.g. ["Minnesota"]
TAG              = None      # output name stem; None -> "conus", or auto from the selection

sel = jobs
if RECIPIENT_STATES != "all":
    want = set(RECIPIENT_STATES)
    unknown = want - set(menu.index)
    assert not unknown, f"not a recipient state in this scenario: {sorted(unknown)}"
    sel = sel[sel.recipients.map(lambda rs: bool(want & set(rs)))]
if SOURCE_STATES != "all":
    want_src = set(SOURCE_STATES)
    unknown = want_src - set(jobs.source)
    assert not unknown, f"not a source state in this scenario: {sorted(unknown)}"
    sel = sel[sel.source.isin(want_src)]
sel = sel.reset_index(drop=True)
assert len(sel), "the filter selected no columns"

if RECIPIENT_STATES == "all" and SOURCE_STATES == "all":
    stem = TAG or "conus"
else:
    picked = (RECIPIENT_STATES if RECIPIENT_STATES != "all" else SOURCE_STATES)
    stem = TAG or "-".join(s.lower().replace(" ", "") for s in sorted(picked))

OUT = TOOLS / f"jobs_{EXPERIMENT.lower()}_{stem}_{SCENARIO}gt.txt"
print(f"{len(sel)} of {len(jobs)} columns -> {2 * len(sel)} runs -> {OUT.name}")

if RECIPIENT_STATES != "all":
    sel_area = sel.area_ha.sum()
    inside   = sum(a for r in sel.itertuples() for s, a in r.by_state.items() if s in want)
    asked    = sum(recip_area[s] for s in want)
    print(f"  {inside/1e6:.2f} of {asked/1e6:.2f} M ha deployed in your states is covered "
          f"({100*inside/asked:.1f}%)")
    print(f"  {(sel_area - inside)/1e6:.2f} M ha of cropland in other states rides along, "
          f"sharing these columns")

338 of 338 columns -> 676 runs -> jobs_e0_conus_2gt.txt


## 6. Cost

Estimate of ~97 s per simulated year at 1 yr, ~110–123 s at 5 yr. All jobs are longer
than 5 yr so this estimate is a lower bound.

In [7]:
SEC_PER_SIM_YEAR = 110      # lower bound; see how-to-run.md 4

results = REPO / "results"
todo = [(r, m) for r in sel.itertuples() for m in (1, 2)
        if not (results / f"{EXPERIMENT}_drv{r.drv}_{r.basalt}_{m}_{r.tf}yr.mat").exists()]
done = 2 * len(sel) - len(todo)

sim_years = sum(r.tf for r, _ in todo)
serial_h  = sim_years * SEC_PER_SIM_YEAR / 3600
longest_h = max(r.tf for r, _ in todo) * SEC_PER_SIM_YEAR / 3600
print(f"{len(todo)} runs outstanding ({done} already in results/), {sim_years:,} simulated years")
print(f"serial: {serial_h:,.0f} h  ({serial_h/24:.1f} days) -- lower bound")
print(f"longest single job: {longest_h:.1f} h ({max(r.tf for r, _ in todo)} yr) -- the floor "
      f"on wall time no matter how wide you go\n")
for n in (8, 16, 32, 64, 128):
    print(f"  {n:3d} workers  {max(serial_h / n, longest_h):6.1f} h")

666 runs outstanding (10 already in results/), 17,282 simulated years
serial: 528 h  (22.0 days) -- lower bound
longest single job: 1.6 h (51 yr) -- the floor on wall time no matter how wide you go

    8 workers    66.0 h
   16 workers    33.0 h
   32 workers    16.5 h
   64 workers     8.3 h
  128 workers     4.1 h


## 8. Write the file

In [8]:
scen_gt = f"{SCENARIO} Gt/yr"
hdr = [
    f"# {EXPERIMENT} at every deployed driver cluster in the {scen_gt} scenario"
    f" ({'CONUS' if stem == 'conus' else stem}),",
    f"# both mineralogy clusters, published source state and start year.",
    f"# {2 * len(sel)} runs, {int(sel.tf.sum()) * 2:,} simulated years total.",
    "#",
    "# Generated by tools/make_jobs_conus.ipynb from",
    f"#   {PICKLE}",
    f"#   {RUN_STRC.relative_to(REPO)}",
    "#",
    "# drv basalt            start  tf  exp",
]

lines = []
for r in sel.sort_values(["tf", "drv"], ascending=[False, True]).itertuples():
    for m in (1, 2):
        lines.append(f"{r.drv:<4} {r.basalt + '_' + str(m):<20} {r.start_year}  {r.tf:<3} {EXPERIMENT}")

OUT.write_text("\n".join(hdr + lines) + "\n")
print(f"wrote {OUT}  ({len(lines)} jobs)\n")
print("\n".join(hdr + lines[:6]))
print("...")
print("\n".join(lines[-2:]))

wrote /Users/tylerkukla/Documents/GitHub/beerling-2025-model-extension/tools/jobs_e0_conus_2gt.txt  (676 jobs)

# E0 at every deployed driver cluster in the 2 Gt/yr scenario (CONUS),
# both mineralogy clusters, published source state and start year.
# 676 runs, 17,546 simulated years total.
#
# Generated by tools/make_jobs_conus.ipynb from
#   /Users/tylerkukla/Downloads/USA_ERW/data/compiled_run_CO2_a_b_v6_varz.pkl
#   upstream/weathering_model/run_strc_6.mat
#
# drv basalt            start  tf  exp
23   Minnesota_1          2020  51  E0
23   Minnesota_2          2020  51  E0
73   Wisconsin_1          2020  51  E0
73   Wisconsin_2          2020  51  E0
84   Michigan_1           2020  51  E0
84   Michigan_2           2020  51  E0
...
132  Wisconsin_1          2068  3   E0
132  Wisconsin_2          2068  3   E0


## 9. Read it back

Parse the file the way `run_batch.sh` and `run_one.sh` will for a round-trip test.

In [9]:
parsed = [l.split() for l in OUT.read_text().splitlines()
          if l.strip() and not l.lstrip().startswith("#")]
assert all(len(p) == 5 for p in parsed), "a line does not have 5 fields"
assert len(parsed) == 2 * len(sel)

back = pd.DataFrame(parsed, columns=["drv", "basalt", "start_year", "tf", "exp"])
back[["drv", "start_year", "tf"]] = back[["drv", "start_year", "tf"]].astype(int)
assert (back.start_year + back.tf == LAST_YEAR + 1).all(), "a window does not end in 2070"
assert set(back.basalt.str[-2:]) == {"_1", "_2"}
assert (back.groupby("drv").size() == 2).all(), "a cluster is missing a mineralogy pair"

print(f"{len(back)} jobs parse cleanly, {back.drv.nunique()} clusters x 2 mineralogies")
print(f"windows {back.tf.min()}-{back.tf.max()} yr, all ending {LAST_YEAR}")
back.head()

676 jobs parse cleanly, 338 clusters x 2 mineralogies
windows 3-51 yr, all ending 2070


,drv,basalt,start_year,tf,exp
0,23,Minnesota_1,2020,51,E0
1,23,Minnesota_2,2020,51,E0
2,73,Wisconsin_1,2020,51,E0
3,73,Wisconsin_2,2020,51,E0
4,84,Michigan_1,2020,51,E0


## 10. How to run

```bash
cd /path/to/beerling-2025-model-extension
nohup tools/run_batch.sh tools/jobs_e0_conus_1gt.txt e0_conus_1gt 4 >/dev/null 2>&1 & disown
```

The last argument is the number of concurrent Octave processes.

# ---